# Revision Practical R01 — Agents, Rules, Search and Games (Step-by-Step Edition)

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **When** | Revision week, before the midterm |
| **Revises** | Practicals P01, P02a, P03a, P04a, P05a and P07 |
| **Related lectures** | Lectures 1–15 and 17 |
| **Duration** | 3 hours (or two sittings: Parts A–C, then Parts D–G) |
| **Marks** | Not marked. This is practice for the midterm, written and practical. |

In the six practicals you built six separate programs: a vacuum agent, a chatbot, a rule
base, a maze solver, an A-star searcher and a tic-tac-toe player. In this revision practical
you meet **every one of those ideas again, on one story with one set of data**. By the end
they join up into one small AI system that reads a request, decides whether it is allowed,
plans a route and carries it out.

## 0. How to use this notebook

1. **Run cells in order**, top to bottom. Click a cell and press **Shift + Enter**.
2. `___` (three underscores) in code is a **blank**. Replace it with your answer and run
   the cell again.
3. After most exercises there is a **✅ You should see** box. Compare it with what your cell
   printed. If it is the same, move on. If it is different, look at your blanks again.
4. Some cells ask you to **predict** first. Write your guess in the blank, then run the next
   cell to see the real answer.
5. Each part starts with a **Remember** box: the few facts from that practical that you must
   know for the exam. Each part ends with **Check yourself** questions in exam style.
6. If you restart the notebook, run every cell from the setup cell down again. The parts
   build on each other: Part G uses code from all the others.

> 💡 **Revision tip.** Before you fill a blank, say out loud which practical it comes from
> and what the line does. If you cannot, open that practical and look. That is the revision.

## 1. Objectives

By the end of this session you will be able to:

1. Run the sense–update–decide–act loop and explain what a belief state adds (P01).
2. Match text against patterns with variables and rewrite what was captured (P02a).
3. Reason forward and backward over a rule base, explain a conclusion, find what is missing,
   and diagnose a fault from conflicts (P03a).
4. State a search problem in four parts and compare breadth-first and depth-first search (P04a).
5. Use step costs and heuristics: lowest-cost-first, greedy, A-star, weighted A-star, branch
   and bound (P05a).
6. Choose a move with minimax, speed it up with alpha-beta, and explain the effect of move
   ordering and a depth limit (P07).
7. Explain how all six fit together in one system.

## 2. The case study: Raahi, the campus courier robot

The university has bought a small delivery robot called **Raahi** (*traveller* in Urdu). It
carries notes, forms and chargers between buildings. Here is the campus as Raahi stores it:

```
#################
#S......#.......#
#.#####.#.#####.#
#.#L..#.#.#..C#.#
#.#.#.#.#.##.##.#
#...#...#.......#
###.#####.#####.#
#...,,,,,.......#
#.#######.###.#.#
#.H.......#..A#.#
#################
```

| Symbol | Meaning | Cost of stepping onto it |
|---|---|---|
| `S` | Raahi's charging dock (the start) | 1 |
| `L` `C` `H` `A` | library, cafeteria, hostel, admin block | 1 |
| `.` | paved path | 1 |
| `,` | the central **lawn**: Raahi's wheels are slow on grass | **3** |
| `#` | a wall or a building | cannot enter |

**The data used in every part:**

| Part | Data | Revises |
|---|---|---|
| A | a corridor of 6 offices in the admin block, with parcels waiting in some | P01 |
| B | five typed requests, such as *"Please take my notes to the admin block."* | P02a |
| C | 8 rules and 7 facts about when Raahi may set off, plus two fault reports | P03a |
| D | the campus map above, searched blind | P04a |
| E | the same map, with lawn costs and a heuristic | P05a |
| F | a tic-tac-toe position from the open-day games table | P07 |
| G | all five requests, handled from start to finish | all |

**One question per part.** Each practical answered one question an intelligent agent faces:

| Part | The question Raahi must answer | Idea from the course |
|---|---|---|
| A | I can see only the office I am in. How do I know when I am finished? | belief state |
| B | What is this person asking me to do? | pattern matching |
| C | Am I allowed to do it? If something goes wrong, what is broken? | rules, chaining, diagnosis |
| D | Which way do I go, with no idea where the goal is? | uninformed search |
| E | Which way is **cheapest**, and can a hint save work? | costs, heuristics, A-star |
| F | How do I play against someone who wants me to lose? | minimax, alpha-beta |

## 3. Setup

**Software:** Python 3.10 or later. Standard library only, so there is nothing to install.

Run the cell below **first, every time you open the notebook.**

In [6]:
import heapq                        # priority queue (Part E)
import random                       # random opponent (Part F)
from itertools import count, combinations

___ = None  # the blank you will replace with your answers
print('Setup done. You are ready to start.')

Setup done. You are ready to start.


---
## Part A — The agent and its belief state (revises P01)

> ### Remember (P01)
>
> • **list** = ordered, can grow (a frontier); **dict** = look-up table (state → value); **set** = fast "have I seen it?" (visited); **tuple** = fixed group, can be a dict key or set member (a position `(row, col)`).
>
> • A **generator** (`yield`) hands out values one at a time and can be used **once**.
>
> • An agent runs **sense → update → decide → act**. The **belief state** is what it remembers. Anything not in the belief state cannot affect what it does.
>
> • An agent is described by four things: **abilities, goals, prior knowledge, experience**.

### A1 — The four data structures, on campus data

Each structure below holds real data from this case study. (The route costs come from Part E.)

In [7]:
to_visit  = ['library', 'hostel']  # list : jobs waiting, in order
# dict : place -> cost from the dock
route_cost = {'library': 8, 'hostel': 13, 'admin block': 28}
done      = {'library'}  # set  : jobs already finished
dock      = (1, 1)  # tuple: (row, col) of the dock

to_visit.append('admin block')
print('jobs waiting     :', to_visit)
print('cost to hostel   :', route_cost['hostel'])
print('library done?    :', 'library' in done)
print('dock is at row   :', dock[0], 'col', dock[1])

jobs waiting     : ['library', 'hostel', 'admin block']
cost to hostel   : 13
library done?    : True
dock is at row   : 1 col 1


**Your turn:** fill in the blanks using the structures above.

In [8]:
cost_admin  = route_cost['admin block']      # look up the admin block's cost
hostel_done = 'hostel' in done               # is the hostel in the done set?
first_job   = to_visit[0]                    # the first job in the list
dock_col    = dock[1]                        # the column of the dock

print(cost_admin, hostel_done, first_job, dock_col)

28 False library 1


**✅ You should see:**

```
28 False library 1
```

### A2 — A generator is used once

The admin block has a corridor of **6 offices**, numbered 0 to 5. A parcel is waiting in some
of them. `offices_with_parcels` is a generator: it hands out the office numbers one by one.

In [9]:
N_OFFICES = 6
# office -> parcel waiting?
PARCELS = {0: False, 1: True, 2: False, 3: True, 4: True, 5: False}

def offices_with_parcels(parcels):
    for office, waiting in parcels.items():
        if waiting:
            yield office

print('as a list:', list(offices_with_parcels(PARCELS)))

as a list: [1, 3, 4]


**✅ You should see:**

```
as a list: [1, 3, 4]
```

**Your turn — predict.** We make **one** generator `g` and call `sum(g)` twice. What do the
two sums give?

In [10]:
first_sum  = 8       # 1 + 3 + 4
second_sum = 0       # careful!

In [11]:
# Run this to reveal the real answer.
g = offices_with_parcels(PARCELS)
print('first sum :', sum(g))
print('second sum:', sum(g))
print('you said  :', first_sum, second_sum)

first sum : 8
second sum: 0
you said  : 8 0


**✅ You should see:**

```
first sum : 8
second sum: 0
you said  : 8 0
```

### A3 — The environment: what Raahi can and cannot see

Raahi starts at office **2**, facing right. At each office it reads the door number and sees
whether a parcel is waiting **there**. It does **not** see the other offices.

In [12]:
class Corridor:
    def __init__(self, parcels, start=2):
        self.parcels = dict(parcels)  # a copy, so PARCELS itself never changes
        self.position = start

    def percept(self):
        # Raahi sees ONLY the door number and whether a parcel is waiting here.
        return (self.position, self.parcels[self.position])

    def act(self, action):
        if action == 'pick':
            self.parcels[self.position] = False
        elif action == 'right' and self.position < len(self.parcels) - 1:
            self.position += 1
        elif action == 'left' and self.position > 0:
            self.position -= 1

env = Corridor(PARCELS)
print('the true state (hidden from Raahi):', env.parcels)
print('what Raahi can see              :', env.percept())

the true state (hidden from Raahi): {0: False, 1: True, 2: False, 3: True, 4: True, 5: False}
what Raahi can see              : (2, False)


**✅ You should see:**

```
the true state (hidden from Raahi): {0: False, 1: True, 2: False, 3: True, 4: True, 5: False}
what Raahi can see              : (2, False)
```

### A4 — The loop with a belief state

The belief state has **two** parts: `checked`, the set of offices Raahi has seen with no
parcel left, and `direction`, the way it is walking. Fill in the **three** blanks.

In [13]:
def run_with_belief(env, max_steps=30):
    checked = set()                  # BELIEF 1: offices seen empty
    direction = 'right'              # BELIEF 2: the way I am walking
    actions = 0
    for t in range(max_steps):
        office, parcel = env.percept()                      # SENSE
        if not parcel:                                      # UPDATE
            checked.add(office)  # blank 1: remember this office
        if parcel:                                          # DECIDE
            action = 'pick'
        elif len(checked) == N_OFFICES:  # blank 2: every office checked?
            action = 'stop'
        else:
            if direction == 'right' and office == N_OFFICES - 1:
                direction = 'left'  # blank 3: turn round at the end
            if direction == 'left' and office == 0:
                direction = 'right'
            action = direction
        print('t=%2d  office=%d  parcel=%-5s  action=%-5s  checked=%s'
              % (t, office, parcel, action, sorted(checked)))
        if action == 'stop':
            break
        env.act(action)                                     # ACT
        actions += 1
    return actions

env = Corridor(PARCELS)
n = run_with_belief(env)
waiting = [o for o, p in env.parcels.items() if p]
print('actions taken:', n, '   parcels still waiting:', waiting)

t= 0  office=2  parcel=False  action=right  checked=[2]
t= 1  office=3  parcel=True   action=pick   checked=[2]
t= 2  office=3  parcel=False  action=right  checked=[2, 3]
t= 3  office=4  parcel=True   action=pick   checked=[2, 3]
t= 4  office=4  parcel=False  action=right  checked=[2, 3, 4]
t= 5  office=5  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 6  office=4  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 7  office=3  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 8  office=2  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 9  office=1  parcel=True   action=pick   checked=[2, 3, 4, 5]
t=10  office=1  parcel=False  action=left   checked=[1, 2, 3, 4, 5]
t=11  office=0  parcel=False  action=stop   checked=[0, 1, 2, 3, 4, 5]
actions taken: 11    parcels still waiting: []


**✅ You should see:**

```
t= 0  office=2  parcel=False  action=right  checked=[2]
t= 1  office=3  parcel=True   action=pick   checked=[2]
t= 2  office=3  parcel=False  action=right  checked=[2, 3]
t= 3  office=4  parcel=True   action=pick   checked=[2, 3]
t= 4  office=4  parcel=False  action=right  checked=[2, 3, 4]
t= 5  office=5  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 6  office=4  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 7  office=3  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 8  office=2  parcel=False  action=left   checked=[2, 3, 4, 5]
t= 9  office=1  parcel=True   action=pick   checked=[2, 3, 4, 5]
t=10  office=1  parcel=False  action=left   checked=[1, 2, 3, 4, 5]
t=11  office=0  parcel=False  action=stop   checked=[0, 1, 2, 3, 4, 5]
actions taken: 11    parcels still waiting: []
```

### A5 — Break the belief state on purpose

Now Raahi has **no memory**. Its decision uses only the current percept: pick if there is a
parcel, go left at the right-hand end, otherwise go right.

**Predict first:** will this agent (a) collect every parcel and stop, (b) collect every parcel
but never stop, or (c) leave a parcel behind and never stop?

In [14]:
prediction_no_memory = 'c'

In [15]:
def reflex_policy(percept):
    office, parcel = percept           # the percept is ALL it has
    if parcel:
        return 'pick'
    if office == N_OFFICES - 1:
        return 'left'
    return 'right'

env = Corridor(PARCELS)
visited_offices = []
for t in range(20):  # a safety limit: it would never stop by itself
    visited_offices.append(env.percept()[0])
    env.act(reflex_policy(env.percept()))
print('offices visited  :', visited_offices)
print('parcels left     :', [o for o, p in env.parcels.items() if p])
print('your prediction  :', prediction_no_memory)

offices visited  : [2, 3, 3, 4, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5]
parcels left     : [1]
your prediction  : c


**✅ You should see:**

```
offices visited  : [2, 3, 3, 4, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5, 4, 5]
parcels left     : [1]
your prediction  : c
```

**Predict once more:** if Raahi (with its belief state) starts at office **0** instead of 2,
how many actions does it take before it stops?

In [16]:
predicted_actions_from_0 = 8

In [17]:
# Run this to reveal the real answer.
env = Corridor(PARCELS, start=0)
real = run_with_belief(env)
print('real:', real, '   you said:', predicted_actions_from_0)

t= 0  office=0  parcel=False  action=right  checked=[0]
t= 1  office=1  parcel=True   action=pick   checked=[0]
t= 2  office=1  parcel=False  action=right  checked=[0, 1]
t= 3  office=2  parcel=False  action=right  checked=[0, 1, 2]
t= 4  office=3  parcel=True   action=pick   checked=[0, 1, 2]
t= 5  office=3  parcel=False  action=right  checked=[0, 1, 2, 3]
t= 6  office=4  parcel=True   action=pick   checked=[0, 1, 2, 3]
t= 7  office=4  parcel=False  action=right  checked=[0, 1, 2, 3, 4]
t= 8  office=5  parcel=False  action=stop   checked=[0, 1, 2, 3, 4, 5]
real: 8    you said: 8


**✅ You should see:**

```
t= 0  office=0  parcel=False  action=right  checked=[0]
t= 1  office=1  parcel=True   action=pick   checked=[0]
t= 2  office=1  parcel=False  action=right  checked=[0, 1]
t= 3  office=2  parcel=False  action=right  checked=[0, 1, 2]
t= 4  office=3  parcel=True   action=pick   checked=[0, 1, 2]
t= 5  office=3  parcel=False  action=right  checked=[0, 1, 2, 3]
t= 6  office=4  parcel=True   action=pick   checked=[0, 1, 2, 3]
t= 7  office=4  parcel=False  action=right  checked=[0, 1, 2, 3, 4]
t= 8  office=5  parcel=False  action=stop   checked=[0, 1, 2, 3, 4, 5]
real: 8    you said: 8
```

### A6 — Describe Raahi in four parts (Lecture 1)

Double-click this cell and fill in each line. Name real variables from the code.

| Part | Raahi in the corridor |
|---|---|
| **Abilities** (what it can do) | move `left` or `right`, `pick` up a parcel, and `stop` (the actions in `env.act` and `run_with_belief`) |
| **Goals** (what it is trying to achieve) | pick up every parcel in the 6 offices and then stop |
| **Prior knowledge** (known before it starts) | there are 6 offices (`N_OFFICES`), the corridor has two ends, and what each action does |
| **Experience** (learned while it runs) | the `checked` set (offices seen empty) and the `direction` it is walking |

### Check yourself — Part A

1. Name the four stages of the agent loop and point to the line of `run_with_belief` for each.
2. Why did the agent with no memory never stop? Use the word *percept* in your answer.
3. Why are positions stored as tuples and not lists when we put them in a set?

**My answers — Check yourself, Part A**

1. Sense: `office, parcel = env.percept()`. Update: `checked.add(office)` (inside `if not parcel`). Decide: the `if parcel: ... elif ... else:` block that sets `action`. Act: `env.act(action)`.
2. It has no memory. Its decision uses only the current percept (office number and parcel yes/no). At office 4 it goes right, at office 5 it goes left, and so on forever, because nothing tells it that it already checked the other offices. It never reaches office 1.
3. A list can be changed, so Python does not allow it inside a set. A tuple cannot be changed, so it can go in a set or be a dict key.

---
## Part B — Reading the request (revises P02a)

> ### Remember (P02a)
>
> • Work on **tokens** (a list of words), not one long string.
>
> • A **pattern** is a list of words with **variables** such as `?X`. A **match** returns the **bindings** dictionary, or `None` if the pattern does not fit.
>
> • A variable in the **middle** needs an **anchor**: the fixed word after it tells it where to stop.
>
> • **Swapping person words** (*my* → *your*) makes the reply sound as if it understood. It understands nothing: it only looks at the **shape** of the sentence.
>
> • A **rewrite system** applies rules until a whole pass changes nothing.

### B1 — The requests, as tokens

These five messages arrived on Raahi's screen this morning.

In [18]:
REQUESTS = [
    'Please take my notes to the admin block.',
    'I need my charger at the hostel',
    'Please take this form to the cafeteria',
    'Please take my file to the main gate',
    'Can you sing a song?',
]

def tokens_of(sentence):
    return sentence.lower().strip('.!?').split()

for s in REQUESTS:
    print(tokens_of(s))

['please', 'take', 'my', 'notes', 'to', 'the', 'admin', 'block']
['i', 'need', 'my', 'charger', 'at', 'the', 'hostel']
['please', 'take', 'this', 'form', 'to', 'the', 'cafeteria']
['please', 'take', 'my', 'file', 'to', 'the', 'main', 'gate']
['can', 'you', 'sing', 'a', 'song']


**✅ You should see:**

```
['please', 'take', 'my', 'notes', 'to', 'the', 'admin', 'block']
['i', 'need', 'my', 'charger', 'at', 'the', 'hostel']
['please', 'take', 'this', 'form', 'to', 'the', 'cafeteria']
['please', 'take', 'my', 'file', 'to', 'the', 'main', 'gate']
['can', 'you', 'sing', 'a', 'song']
```

**Your turn:** pick out single words from the tokens of the first request.

In [19]:
t = tokens_of(REQUESTS[0])
third_word = t[2]          # the word at position 2
last_two   = t[-2:]        # a slice: the last two words
print(third_word, last_two)

my ['admin', 'block']


**✅ You should see:**

```
my ['admin', 'block']
```

### B2 — The P02a matcher: a variable only at the end

This is `match` from P02a, unchanged. **Predict** what each call returns (a dictionary, or
`None` without quotes), then run the reveal cell.

In [20]:
def match(pattern, tokens):
    """Match tokens against a pattern whose LAST part may be a variable."""
    bindings = {}
    for i in range(len(pattern)):
        p = pattern[i]
        if p.startswith('?'):
            bindings[p] = tokens[i:]
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None
    if len(pattern) == len(tokens):
        return bindings
    return None

In [21]:
# a) match(['i', 'need', '?X'], tokens_of('I need my charger at the hostel'))
# b) match(['please', 'take', '?X'], tokens_of('Can you sing a song?'))
predict_a = {'?X': ['my', 'charger', 'at', 'the', 'hostel']}
predict_b = None

In [22]:
# Run this to reveal the real answers.
print('a)', match(['i', 'need', '?X'],
                 tokens_of('I need my charger at the hostel')))
print('b)', match(['please', 'take', '?X'], tokens_of('Can you sing a song?')))
print('you said:', predict_a, predict_b)

a) {'?X': ['my', 'charger', 'at', 'the', 'hostel']}
b) None
you said: {'?X': ['my', 'charger', 'at', 'the', 'hostel']} None


**✅ You should see:**

```
a) {'?X': ['my', 'charger', 'at', 'the', 'hostel']}
b) None
you said: {'?X': ['my', 'charger', 'at', 'the', 'hostel']} None
```

Answer (a) is a problem for Raahi: `?X` swallowed the **place** as well as the item. To get
the item and the place separately we need **two** variables, one of them in the middle.

### B3 — match2: a variable in the middle, with an anchor

This is `match2` from P02a Task 2. Fill in the **two** blanks.

In [23]:
def match2(pattern, tokens):
    """Like match, but a variable may sit anywhere,
    ended by the fixed word after it."""
    bindings = {}
    i = 0                                         # our place in the SENTENCE
    for j in range(len(pattern)):                 # our place in the PATTERN
        p = pattern[j]
        if p.startswith('?'):
            if j == len(pattern) - 1:  # the last part: take everything left
                bindings[p] = tokens[i:]
                return bindings
            anchor = pattern[j + 1]
            if anchor not in tokens[i:]:
                return None
            end = tokens.index(anchor, i)
            bindings[p] = tokens[i:end]  # blank 1: the words up to the anchor
            i = end
        else:
            if i >= len(tokens) or tokens[i] != p:
                return None
            i = i + 1                           # blank 2: move one word along
    if i != len(tokens):
        return None
    return bindings

pattern = ['please', 'take', '?X', 'to', 'the', '?Y']
print(match2(pattern, tokens_of(REQUESTS[0])))
print(match2(pattern, tokens_of(REQUESTS[4])))

{'?X': ['my', 'notes'], '?Y': ['admin', 'block']}
None


**✅ You should see:**

```
{'?X': ['my', 'notes'], '?Y': ['admin', 'block']}
None
```

### B4 — The responder: swap person words, fill the template

Each talk rule is a pattern and a reply template. `read_request` returns **three** things:
the reply, the item and the place. Raahi needs the place for everything that follows.

In [24]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

TALK_RULES = [
    (['please', 'take', '?X', 'to', 'the', '?Y'],
     'OK. I will take {X} to the {Y}.'),
    (['i', 'need', '?X', 'at', 'the', '?Y'],
     'OK. I will bring {X} to the {Y}.'),
]
FALLBACK = 'Sorry, I only understand delivery requests.'

def read_request(sentence):
    tokens = tokens_of(sentence)
    for pattern, template in TALK_RULES:  # the first rule that fits wins
        b = match2(pattern, tokens)
        if b is not None:
            item = ' '.join(swap_person(b['?X']))
            place = ' '.join(b['?Y'])
            return template.format(X=item, Y=place), item, place
    return FALLBACK, None, None

**Predict** the exact reply to the second request, *"I need my charger at the hostel"*.

In [25]:
predicted_reply = 'OK. I will bring your charger to the hostel.'

In [26]:
# Run this to reveal every reply.
for s in REQUESTS:
    reply, item, place = read_request(s)
    print('%-42s -> %s' % (s, reply))
print()
print('you said:', predicted_reply)

Please take my notes to the admin block.   -> OK. I will take your notes to the admin block.
I need my charger at the hostel            -> OK. I will bring your charger to the hostel.
Please take this form to the cafeteria     -> OK. I will take this form to the cafeteria.
Please take my file to the main gate       -> OK. I will take your file to the main gate.
Can you sing a song?                       -> Sorry, I only understand delivery requests.

you said: OK. I will bring your charger to the hostel.


**✅ You should see:**

```
Please take my notes to the admin block.   -> OK. I will take your notes to the admin block.
I need my charger at the hostel            -> OK. I will bring your charger to the hostel.
Please take this form to the cafeteria     -> OK. I will take this form to the cafeteria.
Please take my file to the main gate       -> OK. I will take your file to the main gate.
Can you sing a song?                       -> Sorry, I only understand delivery requests.

you said: OK. I will bring your charger to the hostel.
```

Notice request 4: Raahi **sounds** confident about *the main gate*, but the main gate is not
on its map. The pattern matcher has no idea what a place is. Part C will catch this. The
places Raahi does know:

In [27]:
PLACE_LETTER = {'library': 'L', 'cafeteria': 'C',
                'hostel': 'H', 'admin block': 'A'}

for s in REQUESTS:
    reply, item, place = read_request(s)
    print('%-12s on the map? %s' % (place, place in PLACE_LETTER))

admin block  on the map? True
hostel       on the map? True
cafeteria    on the map? True
main gate    on the map? False
None         on the map? False


**✅ You should see:**

```
admin block  on the map? True
hostel       on the map? True
cafeteria    on the map? True
main gate    on the map? False
None         on the map? False
```

### B5 — A rewrite system: tidying the energy formula

The technician typed Raahi's energy formula as a nested list `[operator, left, right]`:
*energy = paved × 1 + (grass × 3 + waiting × 0)*. Fill in the **three** rewrite rules from
P02a Task 3.

In [28]:
def show_expr(e):
    if not isinstance(e, list):
        return str(e)
    return '(' + show_expr(e[1]) + ' ' + e[0] + ' ' + show_expr(e[2]) + ')'

def simplify_node(e):
    op, left, right = e
    if op == '*' and left == 1:
        return right                       # 1 * x -> x
    if op == '*' and right == 1:
        return left                      # x * 1 -> x
    if op == '+' and right == 0:
        return left                      # x + 0 -> x
    if op == '+' and left == 0:
        return right                       # 0 + x -> x
    if op == '*' and (left == 0 or right == 0):
        return 0                         # x * 0 -> 0
    return e

def simplify_once(e):
    if not isinstance(e, list):
        return e
    # inner boxes first
    before = [e[0], simplify_once(e[1]), simplify_once(e[2])]
    after = simplify_node(before)
    if after != before:
        print('     rule applied:', show_expr(before), '->', show_expr(after))
    return after

def simplify(e):
    print('start :', show_expr(e))
    passes = 1
    while True:
        print(' pass', passes)
        new = simplify_once(e)
        if new == e:                       # a whole pass changed nothing: stop
            print('result:', show_expr(new))
            return new
        e, passes = new, passes + 1

ENERGY = ['+', ['*', 'paved', 1], ['+', ['*', 'grass', 3], ['*', 'waiting', 0]]]
tidy = simplify(ENERGY)

start : ((paved * 1) + ((grass * 3) + (waiting * 0)))
 pass 1
     rule applied: (paved * 1) -> paved
     rule applied: (waiting * 0) -> 0
     rule applied: ((grass * 3) + 0) -> (grass * 3)
 pass 2
result: (paved + (grass * 3))


**✅ You should see:**

```
start : ((paved * 1) + ((grass * 3) + (waiting * 0)))
 pass 1
     rule applied: (paved * 1) -> paved
     rule applied: (waiting * 0) -> 0
     rule applied: ((grass * 3) + 0) -> (grass * 3)
 pass 2
result: (paved + (grass * 3))
```

### Check yourself — Part B

1. What does `match2` return when a pattern does not fit, and why is that better than an
   empty dictionary?
2. Raahi replied *"OK. I will take your file to the main gate."* Does it know where the main
   gate is? What does that tell you about judging a program by its replies?
3. Why does `simplify` need more than one pass in general?

**My answers — Check yourself, Part B**

1. It returns `None`. An empty dictionary `{}` would mean "it matched, with nothing to capture", so `None` makes "no match" clearly different from "match with no variables".
2. No. It only copied words into a template. It knows nothing about the main gate. So a confident reply does not prove understanding; you must check what the program really does.
3. One rewrite can make a new simplification possible. For example `(grass * 3) + (waiting * 0)` first becomes `(grass * 3) + 0`, and only the next pass can turn that into `grass * 3`.

---
## Part C — May Raahi set off? A rule base (revises P03a)

> ### Remember (P03a)
>
> • A **rule** is `(head, [body])`: the head is true **if** every atom in the body is true. Rules are **data**; the inference engine never names a rule.
>
> • **Forward chaining** starts from the facts and derives **everything**. It always stops, because it only adds atoms and there are finitely many.
>
> • **Backward chaining** starts from **one goal** and works back to facts. It needs a **loop guard** if rules refer back to themselves.
>
> • A recorded **proof** answers *why*. *Why not* lists what is missing. **Not proved is not the same as false.**
>
> • A **conflict** is a set of assumables that cannot all be true. A **diagnosis** is a smallest set to give up that breaks **every** conflict.

### C1 — The rule base

Meanings first, as Lecture 4 insists. Then 8 rules. **Fill in rules 6 and 7** from the
English in their comments.

In [29]:
# MEANINGS - every atom, before any rule
#   sender_registered    : the sender is a registered student or member of staff
#   destination_known    : the place in the request is on Raahi's map (Part B)
#   parcel_small         : the item fits in Raahi's box
#   robot_free           : Raahi has no other job
#   battery_ok           : the charging log says the battery is above 30 percent
#   parcel_fragile       : the item is marked fragile
#   destination_upstairs : the receiver's office is on an upper floor
#   receiver_phone_known : the receiver's phone number is on record
#   lift_card            : Raahi has been issued a lift card
#   parcel_valuable      : the item is marked valuable
#   request_valid        : the request can be accepted in principle
#   may_accept           : Raahi may take the item into its box
#   may_dispatch         : Raahi may set off now
#   slow_mode            : drive at half speed
#   send_sms             : text the receiver that Raahi is on the way
#   may_use_lift         : Raahi may call the lift
#   upstairs_delivery    : Raahi can deliver to an upper floor
#   needs_signature      : the receiver must sign on Raahi's screen

KB_RULES = [
    ('request_valid',     ['sender_registered', 'destination_known']),     # 1
    ('may_accept',        ['request_valid', 'parcel_small']),              # 2
    ('may_dispatch',      ['may_accept', 'robot_free', 'battery_ok']),     # 3
    ('slow_mode',         ['may_dispatch', 'parcel_fragile']),             # 4
    ('may_use_lift',      ['may_dispatch', 'lift_card']),                  # 5
    # 6: upstairs_delivery if may_use_lift and destination_upstairs
    ('upstairs_delivery',  ['may_use_lift', 'destination_upstairs']),
    # 7: needs_signature if may_dispatch and parcel_valuable
    ('needs_signature',    ['may_dispatch', 'parcel_valuable']),
    ('send_sms',          ['may_dispatch', 'receiver_phone_known']),       # 8
]

BASE_FACTS = ['sender_registered', 'parcel_small', 'robot_free', 'battery_ok',
              'parcel_fragile', 'destination_upstairs', 'receiver_phone_known']

def facts_for(place):
    """The facts for one request: the base facts, plus destination_known
    if Part B's place is on the map. Here Part B hands over to Part C."""
    facts = list(BASE_FACTS)
    if place in PLACE_LETTER:
        facts.append('destination_known')
    return facts

FACTS = facts_for('admin block')
for head, body in KB_RULES:
    print('%-18s <- %s' % (head, ' AND '.join(body)))
print('facts:', FACTS)

request_valid      <- sender_registered AND destination_known
may_accept         <- request_valid AND parcel_small
may_dispatch       <- may_accept AND robot_free AND battery_ok
slow_mode          <- may_dispatch AND parcel_fragile
may_use_lift       <- may_dispatch AND lift_card
upstairs_delivery  <- may_use_lift AND destination_upstairs
needs_signature    <- may_dispatch AND parcel_valuable
send_sms           <- may_dispatch AND receiver_phone_known
facts: ['sender_registered', 'parcel_small', 'robot_free', 'battery_ok', 'parcel_fragile', 'destination_upstairs', 'receiver_phone_known', 'destination_known']


**✅ You should see:**

```
request_valid      <- sender_registered AND destination_known
may_accept         <- request_valid AND parcel_small
may_dispatch       <- may_accept AND robot_free AND battery_ok
slow_mode          <- may_dispatch AND parcel_fragile
may_use_lift       <- may_dispatch AND lift_card
upstairs_delivery  <- may_use_lift AND destination_upstairs
needs_signature    <- may_dispatch AND parcel_valuable
send_sms           <- may_dispatch AND receiver_phone_known
facts: ['sender_registered', 'parcel_small', 'robot_free', 'battery_ok', 'parcel_fragile', 'destination_upstairs', 'receiver_phone_known', 'destination_known']
```

### C2 — Forward chaining: everything that follows

Fill in the **three** blanks (P03a Step 2).

In [30]:
def forward_chain(rules, facts, verbose=True):
    known = set(facts)
    pass_number = 0
    while True:
        pass_number += 1
        added = False
        for head, body in rules:
            if head not in known and all(b in known for b in body):
                known.add(head)                   # blank 1: the rule fires
                added = True                      # blank 2: something was added
                if verbose:
                    print('pass %d: derived %-14s from %s'
                          % (pass_number, head, body))
        if not added:
            if verbose:
                print('pass %d: nothing new -> stop' % pass_number)
            return known                          # blank 3: everything we know

derived = forward_chain(KB_RULES, FACTS)
print('\nrule heads NOT derived:', sorted({h for h, _ in KB_RULES} - derived))

pass 1: derived request_valid  from ['sender_registered', 'destination_known']
pass 1: derived may_accept     from ['request_valid', 'parcel_small']
pass 1: derived may_dispatch   from ['may_accept', 'robot_free', 'battery_ok']
pass 1: derived slow_mode      from ['may_dispatch', 'parcel_fragile']
pass 1: derived send_sms       from ['may_dispatch', 'receiver_phone_known']
pass 2: nothing new -> stop

rule heads NOT derived: ['may_use_lift', 'needs_signature', 'upstairs_delivery']


**✅ You should see:**

```
pass 1: derived request_valid  from ['sender_registered', 'destination_known']
pass 1: derived may_accept     from ['request_valid', 'parcel_small']
pass 1: derived may_dispatch   from ['may_accept', 'robot_free', 'battery_ok']
pass 1: derived slow_mode      from ['may_dispatch', 'parcel_fragile']
pass 1: derived send_sms       from ['may_dispatch', 'receiver_phone_known']
pass 2: nothing new -> stop

rule heads NOT derived: ['may_use_lift', 'needs_signature', 'upstairs_delivery']
```

**Predict:** request 4 asked for *the main gate*, which is not on the map. Which **new**
atoms (not facts) will forward chaining derive for it? Write a set, e.g. `{'a', 'b'}`, or
`set()` for none.

In [31]:
predicted_new = set()

In [32]:
# Run this to reveal the real answer.
gate_facts = facts_for('main gate')
gate_known = forward_chain(KB_RULES, gate_facts, verbose=False)
print('new atoms derived:', gate_known - set(gate_facts))
print('you said         :', predicted_new)

new atoms derived: set()
you said         : set()


**✅ You should see:**

```
new atoms derived: set()
you said         : set()
```

### C3 — Backward chaining with a proof record

`backward_chain` is P03a's version, given in full: it prints each goal, records the proof,
and has a loop guard. Run it on the goal `slow_mode`.

In [33]:
def backward_chain(rules, facts, goal, proof=None, depth=0, seen=None,
                   verbose=True):
    proof = {} if proof is None else proof
    seen = set() if seen is None else seen
    pad = '  ' * depth
    if verbose: print('%sgoal: %s' % (pad, goal))
    if goal in facts:
        if verbose: print('%s  -> known fact' % pad)
        proof[goal] = 'fact'
        return True, proof
    if goal in seen:                                   # loop guard
        if verbose: print('%s  -> already being proved: giving up' % pad)
        return False, proof
    seen = seen | {goal}
    for head, body in rules:
        if head == goal:
            if verbose: print('%s  trying %s <- %s' % (pad, head, body))
            if all(backward_chain(rules, facts, b, proof, depth + 2, seen,
                                  verbose)[0]
                   for b in body):
                proof[goal] = body
                return True, proof
    if verbose: print('%s  -> cannot establish %s' % (pad, goal))
    return False, proof

ok, proof = backward_chain(KB_RULES, FACTS, 'slow_mode')
print('\nproved:', ok)

goal: slow_mode
  trying slow_mode <- ['may_dispatch', 'parcel_fragile']
    goal: may_dispatch
      trying may_dispatch <- ['may_accept', 'robot_free', 'battery_ok']
        goal: may_accept
          trying may_accept <- ['request_valid', 'parcel_small']
            goal: request_valid
              trying request_valid <- ['sender_registered', 'destination_known']
                goal: sender_registered
                  -> known fact
                goal: destination_known
                  -> known fact
            goal: parcel_small
              -> known fact
        goal: robot_free
          -> known fact
        goal: battery_ok
          -> known fact
    goal: parcel_fragile
      -> known fact

proved: True


**✅ You should see:**

```
goal: slow_mode
  trying slow_mode <- ['may_dispatch', 'parcel_fragile']
    goal: may_dispatch
      trying may_dispatch <- ['may_accept', 'robot_free', 'battery_ok']
        goal: may_accept
          trying may_accept <- ['request_valid', 'parcel_small']
            goal: request_valid
              trying request_valid <- ['sender_registered', 'destination_known']
                goal: sender_registered
                  -> known fact
                goal: destination_known
                  -> known fact
            goal: parcel_small
              -> known fact
        goal: robot_free
          -> known fact
        goal: battery_ok
          -> known fact
    goal: parcel_fragile
      -> known fact

proved: True
```

**Your turn:** read two entries of the proof record.

In [34]:
# proof['may_dispatch'] and proof['battery_ok']
proof_of_dispatch = ['may_accept', 'robot_free', 'battery_ok']
proof_of_battery  = 'fact'
print(proof_of_dispatch == proof['may_dispatch'],
      proof_of_battery == proof['battery_ok'])

True True


**✅ You should see:**

```
True True
```

### C4 — Why, and why not

`why` reads the proof record. `why_not_deep` (P03a Task 3) keeps asking *why not* until it
reaches an atom that no fact and no rule can supply: the **root cause**.

In [35]:
def why(proof, atom):
    if atom not in proof:
        return '%s was never established.' % atom
    if proof[atom] == 'fact':
        return '%s was given as a fact.' % atom
    return '%s because %s.' % (atom, ' and '.join(proof[atom]))

def provable(rules, facts, goal, seen=frozenset()):
    """Quiet backward chaining with a loop guard: True or False."""
    if goal in facts:
        return True
    if goal in seen:
        return False
    return any(head == goal
               and all(provable(rules, facts, b, seen | {goal}) for b in body)
               for head, body in rules)

def why_not_deep(rules, facts, goal, depth=0, seen=frozenset()):
    pad = '    ' * depth
    rules_for_goal = [(h, b) for h, b in rules if h == goal]
    if not rules_for_goal:
        print(pad + '%s: not a fact, and no rule concludes it  <-- ROOT CAUSE'
              % goal)
        return
    for head, body in rules_for_goal:
        missing = [b for b in body
                   if not provable(rules, facts, b, seen | {goal})]
        print(pad + '%s <- %s   missing: %s'
              % (head, ' AND '.join(body), missing))
        for m in missing:
            if m not in seen:
                why_not_deep(rules, facts, m, depth + 1, seen | {goal})

for atom in ['slow_mode', 'may_dispatch', 'battery_ok', 'send_sms']:
    print(why(proof, atom))
print()
why_not_deep(KB_RULES, FACTS, 'upstairs_delivery')

slow_mode because may_dispatch and parcel_fragile.
may_dispatch because may_accept and robot_free and battery_ok.
battery_ok was given as a fact.
send_sms was never established.

upstairs_delivery <- may_use_lift AND destination_upstairs   missing: ['may_use_lift']
    may_use_lift <- may_dispatch AND lift_card   missing: ['lift_card']
        lift_card: not a fact, and no rule concludes it  <-- ROOT CAUSE


**✅ You should see:**

```
slow_mode because may_dispatch and parcel_fragile.
may_dispatch because may_accept and robot_free and battery_ok.
battery_ok was given as a fact.
send_sms was never established.

upstairs_delivery <- may_use_lift AND destination_upstairs   missing: ['may_use_lift']
    may_use_lift <- may_dispatch AND lift_card   missing: ['lift_card']
        lift_card: not a fact, and no rule concludes it  <-- ROOT CAUSE
```

**Your turn:** `send_sms` was derived by forward chaining in C2, yet `why` says it *"was never
established"*. Choose the reason, (a) or (b):

- (a) `send_sms` is false.
- (b) the proof was recorded while proving `slow_mode`, and that proof never needed `send_sms`.

Then name the root cause of `needs_signature` failing (run `why_not_deep` on it if you need to).

In [36]:
reason_send_sms = 'b'
root_cause_signature = 'parcel_valuable'
why_not_deep(KB_RULES, FACTS, 'needs_signature')

needs_signature <- may_dispatch AND parcel_valuable   missing: ['parcel_valuable']
    parcel_valuable: not a fact, and no rule concludes it  <-- ROOT CAUSE


**✅ You should see:**

```
needs_signature <- may_dispatch AND parcel_valuable   missing: ['parcel_valuable']
    parcel_valuable: not a fact, and no rule concludes it  <-- ROOT CAUSE
```

### C5 — Which engine does less work?

Both engines get a counter that goes up by 1 each time a rule is looked at. Just run it.

In [37]:
def forward_count(rules, facts):
    known, examined = set(facts), 0
    while True:
        added = False
        for head, body in rules:
            examined += 1
            if head not in known and all(b in known for b in body):
                known.add(head)
                added = True
        if not added:
            return examined

def backward_count(rules, facts, goal, seen=frozenset(), counter=None):
    counter = [0] if counter is None else counter
    if goal in facts:
        return True, counter[0]
    if goal in seen:
        return False, counter[0]
    for head, body in rules:
        counter[0] += 1
        if head == goal and all(
                backward_count(rules, facts, b, seen | {goal}, counter)[0]
                for b in body):
            return True, counter[0]
    return False, counter[0]

print('%-18s %8s %8s %9s' % ('query', 'proved?', 'forward', 'backward'))
for goal in ['request_valid', 'slow_mode', 'send_sms', 'upstairs_delivery']:
    ok, b = backward_count(KB_RULES, FACTS, goal)
    print('%-18s %8s %8d %9d' % (goal, ok, forward_count(KB_RULES, FACTS), b))

query               proved?  forward  backward
request_valid          True       16         1
slow_mode              True       16        10
send_sms               True       16        14
upstairs_delivery     False       16        30


**✅ You should see:**

```
query               proved?  forward  backward
request_valid          True       16         1
slow_mode              True       16        10
send_sms               True       16        14
upstairs_delivery     False       16        30
```

### C6 — Diagnosis: two fault reports

Raahi was allowed to set off, but two things went wrong:

- it **arrived late** at the admin block;
- its **low-battery alarm** went off, although the charging log says `battery_ok`.

Raahi *assumes* three parts are working: the motor, the map, and the battery sensor. Fill in
the two blanks of the second contradiction rule.

In [38]:
#   ok_motor          : ASSUMABLE - the drive motor works
#   ok_map            : ASSUMABLE - the stored map matches the real campus
#   ok_battery_sensor : ASSUMABLE - the battery sensor reads correctly
#   on_time           : predicted - Raahi arrives on time
#   battery_reads_ok  : predicted - the sensor reports a healthy battery
#   arrived_late      : OBSERVED
#   low_battery_alarm : OBSERVED
#   false             : a contradiction

ASSUMABLES = ['ok_motor', 'ok_map', 'ok_battery_sensor']
OBSERVATIONS = ['arrived_late', 'low_battery_alarm']

DIAG_RULES = KB_RULES + [
    ('on_time',          ['may_dispatch', 'ok_motor', 'ok_map']),
    ('false',            ['on_time', 'arrived_late']),
    ('battery_reads_ok', ['battery_ok', 'ok_battery_sensor']),
    ('false',            ['low_battery_alarm', 'battery_reads_ok']),
]

def derives_false(assumed):
    facts = FACTS + OBSERVATIONS + list(assumed)
    return 'false' in forward_chain(DIAG_RULES, facts, verbose=False)

def find_conflicts(assumables):
    conflicts = []
    for size in range(len(assumables) + 1):
        for group in combinations(assumables, size):
            covered = any(c <= set(group) for c in conflicts)
            if derives_false(group) and not covered:
                conflicts.append(set(group))
    return conflicts

def find_diagnoses(conflicts, assumables):
    diagnoses = []
    for size in range(len(assumables) + 1):
        for group in combinations(assumables, size):
            hits_all = all(set(group) & c for c in conflicts)
            covered = any(d <= set(group) for d in diagnoses)
            if hits_all and not covered:
                diagnoses.append(set(group))
    return diagnoses

**Predict before you run.** How many **conflicts** will there be, and how many assumables
will be in each **diagnosis**?

In [39]:
predicted_conflicts = 2
predicted_diagnosis_size = 2

In [40]:
# Run this to reveal the real answer.
conflicts = find_conflicts(ASSUMABLES)
diagnoses = find_diagnoses(conflicts, ASSUMABLES)
print('conflicts:', [sorted(c) for c in conflicts])
print('diagnoses:', [sorted(d) for d in diagnoses])
print('you said :', predicted_conflicts, 'conflicts, diagnoses of size',
      predicted_diagnosis_size)

conflicts: [['ok_battery_sensor'], ['ok_map', 'ok_motor']]
diagnoses: [['ok_battery_sensor', 'ok_motor'], ['ok_battery_sensor', 'ok_map']]
you said : 2 conflicts, diagnoses of size 2


**✅ You should see:**

```
conflicts: [['ok_battery_sensor'], ['ok_map', 'ok_motor']]
diagnoses: [['ok_battery_sensor', 'ok_motor'], ['ok_battery_sensor', 'ok_map']]
you said : 2 conflicts, diagnoses of size 2
```

### C7 — A circular rule (P03a Task 5, in one cell)

Someone adds *"may_use_lift if lift_booked"* and *"lift_booked if may_use_lift"*. **Predict**
which engine gets stuck: `'forward'`, `'backward without a guard'`, or `'both'`.

In [41]:
predicted_stuck = 'backward without a guard'

In [42]:
CYCLIC = KB_RULES + [('may_use_lift', ['lift_booked']),
                     ('lift_booked', ['may_use_lift'])]

def prove_no_guard(rules, facts, goal):
    if goal in facts:
        return True
    return any(head == goal
               and all(prove_no_guard(rules, facts, b) for b in body)
               for head, body in rules)

try:
    prove_no_guard(CYCLIC, FACTS, 'may_use_lift')
    print('backward, no guard  : finished')
except RecursionError:
    print('backward, no guard  : RecursionError - it never finished')
print('backward, with guard:', provable(CYCLIC, FACTS, 'may_use_lift'))
print('forward chaining    : stopped; may_use_lift derived?',
      'may_use_lift' in forward_chain(CYCLIC, FACTS, verbose=False))
print('you said            :', predicted_stuck)

backward, no guard  : RecursionError - it never finished
backward, with guard: False
forward chaining    : stopped; may_use_lift derived? False
you said            : backward without a guard


**✅ You should see:**

```
backward, no guard  : RecursionError - it never finished
backward, with guard: False
forward chaining    : stopped; may_use_lift derived? False
you said            : backward without a guard
```

### Check yourself — Part C

1. Point to the line of `forward_chain` that names a particular rule. (There is none. Why does
   that matter?)
2. Why does forward chaining always stop, even with the circular rules?
3. The system cannot prove `upstairs_delivery`. Give two different reasons this could happen,
   only one of which means Raahi really must not go upstairs.
4. In C6, explain the difference between the conflict `{ok_map, ok_motor}` and the diagnosis
   `{ok_battery_sensor, ok_map}`.

**My answers — Check yourself, Part C**

1. No line of `forward_chain` names a rule; it just loops over `rules` as data. That matters because we can add or change rules without changing the engine.
2. It only ever adds atoms to `known`, and there is a limited number of atoms. When a pass adds nothing new, it stops. Circular rules cannot start without a fact, so they never fire.
3. (a) Raahi really cannot go upstairs (for example the destination is not upstairs, or the lift is not allowed). (b) The knowledge base is just missing a fact, such as `lift_card`; it was never recorded, so "not proved" does not mean "false". Only (a) is a real ban.
4. A conflict is a set of assumptions that cannot all be true together, e.g. `{ok_map, ok_motor}`. A diagnosis is a smallest set of assumptions to give up so that every conflict is broken, e.g. `{ok_battery_sensor, ok_map}`.

---
## Part D — Finding the way, blind (revises P04a)

> ### Remember (P04a)
>
> • A search problem has four parts: **start state, actions, goal test, cost**.
>
> • The **state** holds everything that affects the future and nothing else: here `(row, col)`. The route travelled is **not** part of the state.
>
> • The frontier holds **paths**. **BFS** takes the **oldest** (`pop(0)`, a queue); **DFS** takes the **newest** (`pop()`, a stack). That one line is the only difference.
>
> • BFS finds the route with the **fewest steps**. DFS returns the first route it stumbles on.
>
> • The **visited** set stops two different paths to the same cell both being expanded.

### D1 — Reading the map

`parse` collects the walls, the lawn cells and the lettered places. Fill in the **two** blanks.

In [43]:
CAMPUS = [
    '#################',
    '#S......#.......#',
    '#.#####.#.#####.#',
    '#.#L..#.#.#..C#.#',
    '#.#.#.#.#.##.##.#',
    '#...#...#.......#',
    '###.#####.#####.#',
    '#...,,,,,.......#',
    '#.#######.###.#.#',
    '#.H.......#..A#.#',
    '#################',
]

def parse(campus):
    walls, grass, places = set(), set(), {}
    for r, row in enumerate(campus):
        for c, ch in enumerate(row):
            if ch == '#':
                walls.add((r, c))             # blank 1: remember this wall
            elif ch == ',':
                grass.add((r, c))
            elif ch.isalpha():                  # a letter: S, L, C, H or A
                places[ch] = (r, c)           # blank 2: where this place is
    return walls, grass, places

WALLS, GRASS, PLACES = parse(CAMPUS)
print('places:', PLACES)
print('walls :', len(WALLS), '  lawn cells:', len(GRASS))

places: {'S': (1, 1), 'L': (3, 3), 'C': (3, 13), 'H': (9, 2), 'A': (9, 13)}
walls : 104   lawn cells: 5


**✅ You should see:**

```
places: {'S': (1, 1), 'L': (3, 3), 'C': (3, 13), 'H': (9, 2), 'A': (9, 13)}
walls : 104   lawn cells: 5
```

### D2 — The four components as code

The only change from P04a is the **cost**: stepping onto lawn costs 3. Fill in the blank.

In [44]:
class CampusProblem:
    def __init__(self, walls, grass, start, goal):
        self.walls, self.grass = walls, grass
        self.start, self.goal = start, goal

    def start_state(self):
        return self.start

    def is_goal(self, state):
        return state == self.goal

    def step_cost(self, cell):
        # blank: 3 on the lawn, 1 elsewhere
        return 3 if cell in self.grass else 1

    def actions(self, state):
        """Yield (next_state, cost) for each legal move:
        up, down, left, right."""
        r, c = state
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nxt = (r + dr, c + dc)
            if nxt not in self.walls:
                yield nxt, self.step_cost(nxt)

def route_cost(problem, path):
    return sum(problem.step_cost(cell) for cell in path[1:])

# dock -> admin block
problem = CampusProblem(WALLS, GRASS, PLACES['S'], PLACES['A'])
print('start', problem.start_state(), '  goal', problem.goal)
print('moves from (7, 3):', list(problem.actions((7, 3))))

start (1, 1)   goal (9, 13)
moves from (7, 3): [((6, 3), 1), ((7, 2), 1), ((7, 4), 3)]


**✅ You should see:**

```
start (1, 1)   goal (9, 13)
moves from (7, 3): [((6, 3), 1), ((7, 2), 1), ((7, 4), 3)]
```

**Your turn:** state the four components of this problem.

In [45]:
d_start   = (1, 1)              # the start state
d_goal    = (9, 13)             # the goal state (the admin block)
d_actions = 'up, down, left, right (not into a wall)'
d_costs   = '1 on paved path, 3 on the lawn'
print(d_start == problem.start_state(), d_goal == problem.goal)

True True


**✅ You should see:**

```
True True
```

### D3 — One loop, two strategies

This is P04a's `search`, renamed `blind_search`. Fill in the **two** blanks that decide the
strategy.

In [46]:
def blind_search(problem, strategy='bfs', use_visited=True):
    frontier = [[problem.start_state()]]          # a list of PATHS
    visited, expanded = set(), 0
    while frontier:
        if strategy == 'bfs':
            path = frontier.pop(0)              # blank 1: the OLDEST path
        else:
            path = frontier.pop()               # blank 2: the NEWEST path
        state = path[-1]
        if use_visited:
            if state in visited:
                continue
            visited.add(state)
        expanded += 1
        if problem.is_goal(state):
            return path, expanded
        for nxt, cost in problem.actions(state):
            if nxt not in path:                   # cycle check
                frontier.append(path + [nxt])
    return None, expanded

print('%-4s %6s %6s %9s' % ('', 'steps', 'cost', 'expanded'))
for strat in ['bfs', 'dfs']:
    path, expanded = blind_search(problem, strat)
    print('%-4s %6d %6d %9d' % (strat.upper(), len(path) - 1,
                                route_cost(problem, path), expanded))

      steps   cost  expanded
BFS      20     30        63
DFS      32     42        68


**✅ You should see:**

```
      steps   cost  expanded
BFS      20     30        63
DFS      32     42        68
```

### D4 — See the routes

`show_map` draws the route as `*` and every other expanded cell as `o`. Place letters stay
visible, and the lawn stays `,` where it was not used.

In [47]:
def show_map(campus, path, explored=(), title=''):
    grid = [list(row) for row in campus]
    for (r, c) in explored:
        if grid[r][c] in '.,':
            grid[r][c] = 'o'
    for (r, c) in path[1:-1]:
        if grid[r][c] in '.,o':             # keep the place letters visible
            grid[r][c] = '*'
    print(title)
    print('\n'.join(''.join(row) for row in grid))
    print()

for strat in ['bfs', 'dfs']:
    path, expanded = blind_search(problem, strat)
    title = '%s: %d steps, cost %d' % (strat.upper(), len(path) - 1,
                                        route_cost(problem, path))
    show_map(CAMPUS, path, (), title)

BFS: 20 steps, cost 30
#################
#S......#.......#
#*#####.#.#####.#
#*#L..#.#.#..C#.#
#*#.#.#.#.##.##.#
#***#...#.......#
###*#####.#####.#
#..***********..#
#.#######.###*#.#
#.H.......#..A#.#
#################

DFS: 32 steps, cost 42
#################
#S******#.......#
#.#####*#.#####.#
#.#L**#*#.#..C#.#
#.#*#*#*#.##.##.#
#..*#***#.......#
###*#####.#####.#
#..***********..#
#.#######.###*#.#
#.H.......#..A#.#
#################



**✅ You should see:**

```
BFS: 20 steps, cost 30
#################
#S......#.......#
#*#####.#.#####.#
#*#L..#.#.#..C#.#
#*#.#.#.#.##.##.#
#***#...#.......#
###*#####.#####.#
#..***********..#
#.#######.###*#.#
#.H.......#..A#.#
#################

DFS: 32 steps, cost 42
#################
#S******#.......#
#.#####*#.#####.#
#.#L**#*#.#..C#.#
#.#*#*#*#.##.##.#
#..*#***#.......#
###*#####.#####.#
#..***********..#
#.#######.###*#.#
#.H.......#..A#.#
#################
```

Read both pictures. **Both** routes cross the lawn (row 7). BFS goes straight down the left
side to reach it. DFS first runs right along the top row, down past the library, and only then
reaches the lawn. **Your turn:**

In [48]:
# moves are pushed up, down, left, right; pop() takes the last
dfs_first_direction = 'right'
bfs_lawn_steps      = 5  # how many lawn cells does the BFS route cross?
# is the BFS route the CHEAPEST route? True or False
bfs_is_cheapest     = False

### D5 — Without the repeated-state check

**Predict:** on this map, will BFS expand `'more'`, `'fewer'` or the `'same'` number of
nodes when `use_visited=False`?

In [49]:
predicted_without_check = 'more'

In [50]:
# Run this to reveal the real answer.
print('%-4s %12s %15s' % ('', 'with check', 'without check'))
for strat in ['bfs', 'dfs']:
    _, w = blind_search(problem, strat, use_visited=True)
    _, wo = blind_search(problem, strat, use_visited=False)
    print('%-4s %12d %15d' % (strat.upper(), w, wo))
print('you said:', predicted_without_check)

       with check   without check
BFS            63              85
DFS            68              92
you said: more


**✅ You should see:**

```
       with check   without check
BFS            63              85
DFS            68              92
you said: more
```

### Check yourself — Part D

1. Quote the one line that makes `blind_search` breadth-first, and the one that makes it
   depth-first.
2. Why is the route travelled not part of the state? What would the visited set do if it were?
3. BFS is called *optimal*. In what sense was its route optimal here, and in what sense was it
   not?

**My answers — Check yourself, Part D**

1. BFS: `path = frontier.pop(0)`. DFS: `path = frontier.pop()`.
2. The state is only what affects the future, and the route does not. If the route were in the state, two different routes to the same cell would be different states, so the visited set would never see a repeat.
3. It was optimal for the fewest steps (20). It was not optimal for cost, because it walked across the lawn (cost 30) while a longer route round the lawn cost only 28.

---
## Part E — The cheapest way, and a hint that saves work (revises P05a)

> ### Remember (P05a)
>
> • **g** = cost already spent (a fact). **h** = estimate of the cost still to go (a guess).
>
> • A heuristic comes from **relaxation**: delete a rule, solve the easier problem exactly.
>
> • **Branch and bound** goes depth-first, keeps the best route so far as a **bound**, and prunes any path whose `g + h` cannot beat it. It uses little memory.

| Strategy | Priority (smallest first) | Cheapest route guaranteed? |
|---|---|---|
| Lowest-cost-first (UCS) | `g` | yes |
| Greedy best-first | `h` | no |
| A-star | `g + h` | yes, if `h` is **admissible** (never overestimates) |
| Weighted A-star | `g + w × h`, `w > 1` | no, and it fails **silently** |

### E1 — The heuristic, by relaxation

Relax **two** rules: walls can be walked through, and the lawn costs 1 like paving. The
cheapest route is then the **grid distance**. Fill in the blank.

In [51]:
def manhattan(state, goal):
    """Relaxed problem: no walls and no lawn penalty.
    Rows to go + columns to go."""
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])

print('h at the dock:', manhattan(PLACES['S'], PLACES['A']))

h at the dock: 20


**✅ You should see:**

```
h at the dock: 20
```

### E2 — g, h and f by hand

Cell `(7, 9)` is just after the lawn. Raahi can reach it in **two different ways**:

| Label | Cell | How Raahi got there | g |
|---|---|---|---|
| P | `(7, 3)` | down the left side to the lawn's edge | 8 |
| Q-lawn | `(7, 9)` | from P straight across the lawn: 5 lawn steps and 1 paved | 8 + 15 + 1 = 24 |
| Q-paved | `(7, 9)` | from P round by the hostel path: 14 paved steps | 8 + 14 = 22 |

Work out **h** (to the admin block `(9, 13)`) and **f = g + h** for each.

In [52]:
h_P = 12            # |7 - 9| + |3 - 13|
f_P = 20
h_Q = 6             # the same cell, so the same h for both routes
f_Q_lawn  = 30
f_Q_paved = 28

better_route_to_Q = 'paved'     # 'lawn' or 'paved': which has the smaller f?

In [53]:
# Run this to reveal the real values.
for name, cell, g in [('P', (7, 3), 8), ('Q-lawn', (7, 9), 24),
                      ('Q-paved', (7, 9), 22)]:
    h = manhattan(cell, PLACES['A'])
    print('%-8s g=%2d  h=%2d  f=%2d' % (name, g, h, g + h))

P        g= 8  h=12  f=20
Q-lawn   g=24  h= 6  f=30
Q-paved  g=22  h= 6  f=28


**✅ You should see:**

```
P        g= 8  h=12  f=20
Q-lawn   g=24  h= 6  f=30
Q-paved  g=22  h= 6  f=28
```

### E3 — One searcher, three strategies

P05a's `search`, renamed `smart_search`. Fill in the **four** blanks.

In [54]:
def smart_search(problem, strategy, heuristic=manhattan, weight=1.0):
    """strategy: 'ucs' | 'greedy' | 'astar'.
    weight multiplies h (A-star only)."""
    ticket = count()
    start = problem.start_state()

    def priority(g, state):
        h = heuristic(state, problem.goal)
        if strategy == 'ucs':
            return g                          # blank 1: cost so far only
        if strategy == 'greedy':
            return h                          # blank 2: estimate only
        return g + weight * h                 # blank 3: A-star

    frontier = [(priority(0, start), next(ticket), 0, [start])]
    visited, expanded = set(), 0
    while frontier:
        # blank 4: the SMALLEST priority
        _, _, g, path = heapq.heappop(frontier)
        state = path[-1]
        if state in visited:
            continue
        visited.add(state)
        expanded += 1
        if problem.is_goal(state):
            return path, g, expanded, visited
        for nxt, cost in problem.actions(state):
            if nxt not in visited:
                ng = g + cost
                item = (priority(ng, nxt), next(ticket), ng, path + [nxt])
                heapq.heappush(frontier, item)
    return None, None, expanded, visited

bfs_path, bfs_expanded = blind_search(problem, 'bfs')
print('%-12s %6s %6s %9s' % ('strategy', 'steps', 'cost', 'expanded'))
print('%-12s %6d %6d %9d' % ('BFS (D3)', len(bfs_path) - 1,
                             route_cost(problem, bfs_path), bfs_expanded))
for strat in ['ucs', 'greedy', 'astar']:
    path, cost, expanded, explored = smart_search(problem, strat)
    print('%-12s %6d %6d %9d' % (strat, len(path) - 1, cost, expanded))

strategy      steps   cost  expanded
BFS (D3)         20     30        63
ucs              28     28        63
greedy           20     30        23
astar            28     28        50


**✅ You should see:**

```
strategy      steps   cost  expanded
BFS (D3)         20     30        63
ucs              28     28        63
greedy           20     30        23
astar            28     28        50
```

Read the table **row by row**:

- **UCS** walks 8 more steps than BFS but pays less (28 against 30): it goes round the lawn.
- **A-star** finds the same cheapest route as UCS with fewer nodes (50 against 63).
- **Greedy** is the fastest (23 nodes) and walks straight across the lawn, because every lawn
  step lowers `h`. It never notices the lawn's cost.

In [55]:
for strat in ['astar', 'greedy']:
    path, cost, expanded, explored = smart_search(problem, strat)
    title = '%s: cost %d, %d nodes expanded' % (strat.upper(), cost, expanded)
    show_map(CAMPUS, path, explored, title)

ASTAR: cost 28, 50 nodes expanded
#################
#Soooooo#.......#
#*#####o#.#####.#
#*#Loo#o#.#..C#.#
#*#o#o#o#.##.##.#
#***#ooo#.......#
###*#####.#####.#
#***oooo,*****..#
#*#######*###*#.#
#*H*******#..A#.#
#################

GREEDY: cost 30, 23 nodes expanded
#################
#S......#.......#
#*#####.#.#####.#
#*#L..#.#.#..C#.#
#*#.#.#.#.##.##.#
#***#...#.......#
###*#####.#####.#
#..***********..#
#.#######o###*#.#
#.H......o#..A#.#
#################



**✅ You should see:**

```
ASTAR: cost 28, 50 nodes expanded
#################
#Soooooo#.......#
#*#####o#.#####.#
#*#Loo#o#.#..C#.#
#*#o#o#o#.##.##.#
#***#ooo#.......#
###*#####.#####.#
#***oooo,*****..#
#*#######*###*#.#
#*H*******#..A#.#
#################

GREEDY: cost 30, 23 nodes expanded
#################
#S......#.......#
#*#####.#.#####.#
#*#L..#.#.#..C#.#
#*#.#.#.#.##.##.#
#***#...#.......#
###*#####.#####.#
#..***********..#
#.#######o###*#.#
#.H......o#..A#.#
#################
```

### E4 — Is the heuristic admissible? Check every cell

`true_cost_from` runs UCS from a cell to find its real cheapest cost to the admin block. Fill
in the blank, which counts the cells where `h` is **bigger** than the true cost.

In [56]:
def true_cost_from(cell):
    sub = CampusProblem(WALLS, GRASS, cell, PLACES['A'])
    return smart_search(sub, 'ucs')[1]

open_cells = [(r, c) for r in range(len(CAMPUS)) for c in range(len(CAMPUS[0]))
              if (r, c) not in WALLS]
goal_A = PLACES['A']
too_big = [cell for cell in open_cells
           if manhattan(cell, goal_A) > true_cost_from(cell)]
exact = [cell for cell in open_cells
         if manhattan(cell, goal_A) == true_cost_from(cell)]
print('open cells checked     :', len(open_cells))
print('h overestimates at     :', len(too_big), 'cells')
print('h is exactly right at  :', len(exact), 'cells')

open cells checked     : 83
h overestimates at     : 0 cells
h is exactly right at  : 24 cells


**✅ You should see:**

```
open cells checked     : 83
h overestimates at     : 0 cells
h is exactly right at  : 24 cells
```

**Write the four-sentence justification (Lecture 12).**

1. I relaxed two rules: **walls cannot block me (I can walk through them)** and **the lawn costs 1 like paving**.
2. In the relaxed problem the cheapest route from `(r, c)` costs **|row difference| + |column difference| (the grid distance)**.
3. It cannot overestimate because every real route is **at least as long as the straight grid distance (walls only force detours)**, and in the real problem each step costs **at least 1** (1 on paving, 3 on the lawn).
4. It is useful because **A-star with this hint expanded only 50 nodes, while UCS without a hint expanded 63, and both found the same cheapest route (cost 28)**.

### E5 — Weighted A-star and the silent failure

Fill in the blank so each run uses the weight `w` of the loop.

In [57]:
print('%-7s %5s %9s %9s' % ('weight', 'cost', 'expanded', 'optimal?'))
for w in [1.0, 1.5, 2.0, 3.0]:
    path, cost, expanded, explored = smart_search(problem, 'astar', weight=w)
    verdict = 'yes' if cost == 28 else 'NO'
    print('%-7.1f %5d %9d %9s' % (w, cost, expanded, verdict))

weight   cost  expanded  optimal?
1.0        28        50       yes
1.5        28        45       yes
2.0        30        25        NO
3.0        30        23        NO


**✅ You should see:**

```
weight   cost  expanded  optimal?
1.0        28        50       yes
1.5        28        45       yes
2.0        30        25        NO
3.0        30        23        NO
```

### E6 — Depth-first branch and bound

P05a Task 5's `branch_and_bound`. Fill in the **three** blanks.

In [58]:
def branch_and_bound(problem, heuristic=manhattan):
    best = {'cost': float('inf'), 'path': None}  # the bound starts at infinity
    counts = {'expanded': 0, 'pruned': 0}

    def dive(path, g):
        state = path[-1]
        counts['expanded'] += 1
        if problem.is_goal(state):
            if g < best['cost']:  # blank 1: better than the best so far?
                best['cost'] = g  # blank 2: tighten the bound
                best['path'] = list(path)
                print('  new bound:', best['cost'])
            return
        for nxt, cost in problem.actions(state):
            if nxt in path:
                continue
            ng = g + cost
            # blank 3: the least it can cost
            if ng + heuristic(nxt, problem.goal) >= best['cost']:
                counts['pruned'] += 1
                continue
            dive(path + [nxt], ng)

    dive([problem.start_state()], 0)
    return best['path'], best['cost'], counts['expanded'], counts['pruned']

bb_path, bb_cost, bb_expanded, bb_pruned = branch_and_bound(problem)
print('final cost', bb_cost, '  nodes expanded', bb_expanded,
      '  branches pruned', bb_pruned)

  new bound: 44
  new bound: 36
  new bound: 28
final cost 28   nodes expanded 108   branches pruned 8


**✅ You should see:**

```
  new bound: 44
  new bound: 36
  new bound: 28
final cost 28   nodes expanded 108   branches pruned 8
```

### E7 — Your turn: the cafeteria

Request 3 goes to the **cafeteria** `C`. **Predict** whether BFS's route will be the
cheapest there too, then run the cell.

In [59]:
predicted_bfs_cheapest_to_C = False

In [60]:
to_cafe = CampusProblem(WALLS, GRASS, PLACES['S'], PLACES['C'])
p, _ = blind_search(to_cafe, 'bfs')
print('BFS    : %d steps, cost %d' % (len(p) - 1, route_cost(to_cafe, p)))
for strat in ['ucs', 'greedy', 'astar']:
    p, cost, expanded, _ = smart_search(to_cafe, strat)
    print('%-7s: %d steps, cost %d, %d expanded'
          % (strat, len(p) - 1, cost, expanded))
print('you said:', predicted_bfs_cheapest_to_C)

BFS    : 22 steps, cost 32
ucs    : 30 steps, cost 30, 73 expanded
greedy : 34 steps, cost 44, 42 expanded
astar  : 30 steps, cost 30, 59 expanded
you said: False


**✅ You should see:**

```
BFS    : 22 steps, cost 32
ucs    : 30 steps, cost 30, 73 expanded
greedy : 34 steps, cost 44, 42 expanded
astar  : 30 steps, cost 30, 59 expanded
you said: False
```

### Check yourself — Part E

1. In plain words, what are g, h and f for the cell `(7, 9)` on the paved route?
2. Show the one line that turns `smart_search` into greedy search, and explain why greedy
   walked across the lawn.
3. If `manhattan` returned 0 everywhere, which algorithm would `smart_search(..., 'astar')`
   become? How many nodes would it expand here?
4. What did weight 2 buy, and what did it cost? Why did the program not warn you?

**My answers — Check yourself, Part E**

1. g = the cost already spent to get to `(7, 9)` (22 on the paved route). h = the guess of the cost still to go to the admin block (6). f = g + h = 28.
2. `return h` in the `greedy` branch of `priority`. It walked across the lawn because each lawn step makes h smaller, and greedy ignores the real cost (3 per lawn step).
3. It would become uniform-cost search (lowest-cost-first), because the priority is just g. It would expand 63 nodes here.
4. Weight 2 bought speed (25 nodes instead of 50) but cost optimality (cost 30 instead of 28). The program did not warn because the search still finds a valid route; it cannot tell that a cheaper one exists. It fails silently.

---
## Part F — The open-day games table (revises P07)

At the university open day, Raahi plays tic-tac-toe against visitors. Raahi is **X** and
**maximises**; the visitor is **O** and **minimises**.

> ### Remember (P07)
>
> • A game needs a **strategy** (a reply to every opponent move), not a plan.
>
> • **Minimax**: MAX takes the largest child value, MIN the smallest; values come up from the leaves. It assumes the opponent plays perfectly, so its value is a **guaranteed floor**.
>
> • **Alpha-beta** returns **exactly** the same value and move. It skips branches that cannot change the decision. How much it skips depends on **move ordering**.
>
> • Big games need a **depth limit** and an **evaluation function**. A short horizon can miss a threat just beyond it.

### F1 — A game tree by hand

Before a game, Raahi has 3 possible opening moves. For each, the visitor has 3 replies. The
leaves are scores for Raahi:

```
                 Raahi (MAX)
        /             |             \
     move a         move b         move c        <- visitor (MIN) chooses
    /  |  \        /  |  \        /  |  \
   3  12   8      2   4   6     14   5   2
```

**By hand:** the MIN value of each move, and the root value.

In [61]:
min_a = 3
min_b = 2
min_c = 2
root_value = 3
best_opening = 'a'

Now alpha-beta, reading leaves left to right. It records every leaf it **looks at**. Before
you run it: which **two** leaves will it never look at?

In [62]:
predicted_skipped = [4, 6]

In [63]:
TREE = [[3, 12, 8], [2, 4, 6], [14, 5, 2]]

def alphabeta_tree(tree):
    alpha, looked_at = float('-inf'), []
    for replies in tree:                    # each of Raahi's moves
        beta = float('inf')
        for leaf in replies:                # the visitor's replies
            looked_at.append(leaf)
            beta = min(beta, leaf)
            if beta <= alpha:  # MIN can hold Raahi to <= alpha: stop looking
                break
        alpha = max(alpha, beta)
    return alpha, looked_at

value, looked = alphabeta_tree(TREE)
plain = max(min(r) for r in TREE)
print('root value      :', value, '  (plain minimax:', plain, ')')
print('leaves looked at:', looked, '->', len(looked), 'of 9')
print('you said skipped:', predicted_skipped)

root value      : 3   (plain minimax: 3 )
leaves looked at: [3, 12, 8, 2, 14, 5, 2] -> 7 of 9
you said skipped: [4, 6]


**✅ You should see:**

```
root value      : 3   (plain minimax: 3 )
leaves looked at: [3, 12, 8, 2, 14, 5, 2] -> 7 of 9
you said skipped: [4, 6]
```

**Move ordering.** Suppose the visitor's replies were looked at **best for the visitor first**
(smallest first). Predict how many leaves alpha-beta would look at.

In [64]:
predicted_leaves_sorted = 5

In [65]:
SORTED = [sorted(replies) for replies in TREE]
value, looked = alphabeta_tree(SORTED)
print('tree           :', SORTED)
print('root value     :', value)
print('leaves looked  :', looked, '->', len(looked), 'of 9')
print('you said       :', predicted_leaves_sorted)

tree           : [[3, 8, 12], [2, 4, 6], [2, 5, 14]]
root value     : 3
leaves looked  : [3, 8, 12, 2, 2] -> 5 of 9
you said       : 5


**✅ You should see:**

```
tree           : [[3, 8, 12], [2, 4, 6], [2, 5, 14]]
root value     : 3
leaves looked  : [3, 8, 12, 2, 2] -> 5 of 9
you said       : 5
```

### F2 — A real position

The P07 tic-tac-toe code, given. It is Raahi's (X's) turn on this board:

In [66]:
EMPTY = ' '
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
MOVE_ORDER = 'normal'      # 'reversed' tries the cells from 8 down to 0

def moves(board):
    m = [i for i, c in enumerate(board) if c == EMPTY]
    return m[::-1] if MOVE_ORDER == 'reversed' else m

def play(board, i, player):
    b = list(board); b[i] = player; return tuple(b)

def winner(board):
    for a, b, c in LINES:
        if board[a] != EMPTY and board[a] == board[b] == board[c]:
            return board[a]
    return None

def terminal(board):
    return winner(board) is not None or not moves(board)

def show_board(board):
    for r in range(3):
        cells = ' | '.join(board[3*r:3*r+3])
        numbers = ' | '.join(str(i) for i in range(3*r, 3*r+3))
        print(' ' + cells + '        ' + numbers)
        if r < 2: print(' ---+---+---       ---+---+---')

BOARD = (' ', ' ', 'X',
         'X', ' ', ' ',
         ' ', 'O', 'O')
show_board(BOARD)

   |   | X        0 | 1 | 2
 ---+---+---       ---+---+---
 X |   |          3 | 4 | 5
 ---+---+---       ---+---+---
   | O | O        6 | 7 | 8


**✅ You should see:**

```
   |   | X        0 | 1 | 2
 ---+---+---       ---+---+---
 X |   |          3 | 4 | 5
 ---+---+---       ---+---+---
   | O | O        6 | 7 | 8
```

**Your turn:** which cell **must** Raahi take now, and why?

In [67]:
must_play = 6     # the cell number

### F3 — Minimax and alpha-beta on this position

Fill in the **two** blanks: X takes the `max`, O takes the `min`.

In [5]:
nodes_examined = 0

def value(board, player):
    """Minimax value from X's point of view: +1 X wins, -1 O wins, 0 draw."""
    global nodes_examined
    nodes_examined += 1
    w = winner(board)
    if w == 'X': return 1
    if w == 'O': return -1
    if not moves(board): return 0
    if player == 'X':
        return max(value(play(board, m, 'X'), 'O') for m in moves(board))
    return min(value(play(board, m, 'O'), 'X') for m in moves(board))

def value_ab(board, player, alpha=-2, beta=2):
    global nodes_examined
    nodes_examined += 1
    w = winner(board)
    if w == 'X': return 1
    if w == 'O': return -1
    if not moves(board): return 0
    if player == 'X':
        best = -2
        for m in moves(board):
            best = max(best, value_ab(play(board, m, 'X'), 'O', alpha, beta))
            alpha = max(alpha, best)
            if beta <= alpha:
                break  # this branch is already refuted
        return best
    best = 2
    for m in moves(board):
        best = min(best, value_ab(play(board, m, 'O'), 'X', alpha, beta))
        beta = min(beta, best)
        if beta <= alpha:
            break
    return best

def best_move(board, player):
    opts = moves(board)
    if player == 'X':
        return max(opts, key=lambda m: value_ab(play(board, m, 'X'), 'O'))
    return min(opts, key=lambda m: value_ab(play(board, m, 'O'), 'X'))

values = {m: value_ab(play(BOARD, m, 'X'), 'O') for m in moves(BOARD)}
print('value of each move for X:', values)
print('best move for X         :', best_move(BOARD, 'X'))
print()
print('%-28s %6s %6s' % ('method', 'value', 'nodes'))
for name, fn, order in [('plain minimax', value, 'normal'),
                        ('alpha-beta, normal order', value_ab, 'normal'),
                        ('alpha-beta, reversed order', value_ab, 'reversed')]:
    MOVE_ORDER = order
    nodes_examined = 0
    v = fn(BOARD, 'X')
    print('%-28s %6d %6d' % (name, v, nodes_examined))
MOVE_ORDER = 'normal'

NameError: name 'moves' is not defined

**✅ You should see:**

```
value of each move for X: {0: -1, 1: -1, 4: -1, 5: -1, 6: 1}
best move for X         : 6

method                        value  nodes
plain minimax                     1    150
alpha-beta, normal order          1     83
alpha-beta, reversed order        1     37
```

### F4 — Can Raahi be beaten?

Raahi plays 20 games from the empty board against a visitor who moves at random. Just run it
(it takes a few seconds).

In [64]:
random.seed(0)
results = {'X wins': 0, 'O wins': 0, 'draws': 0}
for game in range(20):
    board, player = tuple([EMPTY] * 9), 'X'
    while not terminal(board):
        if player == 'X':
            m = best_move(board, 'X')
        else:
            m = random.choice(moves(board))
        board = play(board, m, player)
        player = 'O' if player == 'X' else 'X'
    w = winner(board)
    results['X wins' if w == 'X' else 'O wins' if w == 'O' else 'draws'] += 1
print(results)

{'X wins': 20, 'O wins': 0, 'draws': 0}


**✅ You should see:**

```
{'X wins': 20, 'O wins': 0, 'draws': 0}
```

**Predict:** if the visitor also played perfectly (minimax), what would every game end in?

In [65]:
perfect_vs_perfect = 'draw'

### F5 — A depth limit and an evaluation function

Real games are too big to search to the end, so we stop at a depth and **estimate**. The
evaluation counts the lines still open for X minus the lines still open for O. Fill in the
blank, then compare depth 1 with depth 2 on `BOARD`.

In [4]:
def open_lines(board, player):
    other = 'O' if player == 'X' else 'X'
    return sum(1 for line in LINES if all(board[i] != other for i in line))

def evaluate(board):
    return open_lines(board, 'X') - open_lines(board, 'O')

def value_limited(board, player, depth):
    w = winner(board)
    if w == 'X': return 100
    if w == 'O': return -100
    if not moves(board): return 0
    if depth == 0:
        return evaluate(board)                      # stop here and estimate
    if player == 'X':
        return max(value_limited(play(board, m, 'X'), 'O', depth - 1)
                   for m in moves(board))
    return min(value_limited(play(board, m, 'O'), 'X', depth - 1)
               for m in moves(board))

for depth in [1, 2]:
    scores = {m: value_limited(play(BOARD, m, 'X'), 'O', depth - 1)
              for m in moves(BOARD)}
    choice = max(scores, key=scores.get)
    print('depth %d scores: %s   -> chooses %d' % (depth, scores, choice))

NameError: name 'moves' is not defined

**✅ You should see:**

```
depth 1 scores: {0: 2, 1: 2, 4: 3, 5: 1, 6: 2}   -> chooses 4
depth 2 scores: {0: -100, 1: -100, 4: -100, 5: -100, 6: 0}   -> chooses 6
```

### Check yourself — Part F

1. Why does the opponent minimise rather than maximise?
2. In F1, why could alpha-beta skip the leaves 4 and 6 without any risk?
3. Alpha-beta looked at 83 nodes in the normal order and 37 in the reversed order. Did either
   give a better move? Why not?
4. At depth 1 Raahi chose a losing move. Was the evaluation function wrong, or the depth too
   small? Explain.

**My answers — Check yourself, Part F**

1. The visitor wants Raahi to lose, so the visitor picks the move that is worst for Raahi, which is the smallest score for Raahi.
2. Once Raahi knows move a is worth at least 3, and the visitor can answer move b with a 2, move b can never be better than 2. So whatever the other leaves (4, 6) are, move b will not be chosen.
3. No. Both gave value 1 and the same best move. Alpha-beta only skips branches that cannot change the result; move ordering changes how much work is skipped, not the answer.
4. The depth was too small. The evaluation counted open lines and liked cell 4, but it could not see that O wins next move. With depth 2 it saw the threat and chose cell 6.

---
## Part G — Putting it all together

`deliver` handles one typed request from start to finish, using one piece from each part:

| Stage | Code used | From |
|---|---|---|
| 1. read the request | `read_request` | Part B (P02a) |
| 2. may Raahi set off? | `facts_for`, `forward_chain`, `why_not_deep` | Part C (P03a) |
| 3. plan the cheapest route | `smart_search(..., 'astar')` | Parts D–E (P04a, P05a) |
| 4. drive it, keeping a belief state | sense → update → decide → act | Part A (P01) |

Fill in the **three** blanks.

In [3]:
def deliver(sentence):
    print('REQUEST :', sentence)
    reply, item, place = read_request(sentence)  # 1. Part B
    print('RAAHI   :', reply)
    if place is None:
        return 'no delivery'

    facts = facts_for(place)  # 2. Part C
    known = forward_chain(KB_RULES, facts, verbose=False)
    if 'may_dispatch' not in known:
        print('REFUSED : I may not set off. Why not:')
        why_not_deep(KB_RULES, facts, 'may_dispatch')
        return 'refused'
    if 'slow_mode' in known:
        print('NOTE    : fragile item, driving at half speed')

    goal = PLACES[PLACE_LETTER[place]]  # 3. Parts D-E
    trip = CampusProblem(WALLS, GRASS, PLACES['S'], goal)
    path, cost, expanded, _ = smart_search(trip, 'astar')

    battery = 100  # 4. Part A
    position = path[0]  # BELIEF STATE: where I am, and my charge
    for planned in path[1:]:
        position = planned                   # ACT: one step along the plan
        # UPDATE: subtract what this step cost
        battery -= trip.step_cost(planned)
    print('ARRIVED : %s, %d steps, cost %d, battery now %d%% '
          '(A-star expanded %d nodes)'
          % (place, len(path) - 1, cost, battery, expanded))
    return 'delivered'

summary = []
for s in REQUESTS:
    summary.append(deliver(s))
    print()
print('summary:', summary)

NameError: name 'REQUESTS' is not defined

**✅ You should see:**

```
REQUEST : Please take my notes to the admin block.
RAAHI   : OK. I will take your notes to the admin block.
NOTE    : fragile item, driving at half speed
ARRIVED : admin block, 28 steps, cost 28, battery now 72% (A-star expanded 50 nodes)

REQUEST : I need my charger at the hostel
RAAHI   : OK. I will bring your charger to the hostel.
NOTE    : fragile item, driving at half speed
ARRIVED : hostel, 13 steps, cost 13, battery now 87% (A-star expanded 18 nodes)

REQUEST : Please take this form to the cafeteria
RAAHI   : OK. I will take this form to the cafeteria.
NOTE    : fragile item, driving at half speed
ARRIVED : cafeteria, 30 steps, cost 30, battery now 70% (A-star expanded 59 nodes)

REQUEST : Please take my file to the main gate
RAAHI   : OK. I will take your file to the main gate.
REFUSED : I may not set off. Why not:
may_dispatch <- may_accept AND robot_free AND battery_ok   missing: ['may_accept']
    may_accept <- request_valid AND parcel_small   missing: ['request_valid']
        request_valid <- sender_registered AND destination_known   missing: ['destination_known']
            destination_known: not a fact, and no rule concludes it  <-- ROOT CAUSE

REQUEST : Can you sing a song?
RAAHI   : Sorry, I only understand delivery requests.

summary: ['delivered', 'delivered', 'delivered', 'refused', 'no delivery']
```

### Check yourself — Part G

1. Request 4 got a confident reply and was then refused. Which part made each decision, and
   which part was right?
2. Why does `deliver` use A-star and not BFS to plan the route?
3. Name one thing Raahi's belief state holds while driving, and one thing it would need to
   hold if a corridor could be blocked by a crowd.

**My answers — Check yourself, Part G**

1. Part B (the pattern matcher) made the confident reply, and Part C (the rules) made the refusal. Part C was right, because the main gate is not on the map.
2. A-star finds the cheapest route (28) with fewer nodes, while BFS only finds the fewest steps and ignores the lawn cost.
3. It holds its position and its battery level. If a corridor could be blocked by a crowd, it would also need to remember which cells are blocked.

---
## One-page summary: the line that matters

| Practical | Idea | The line of code that carries it |
|---|---|---|
| P01 | belief state | `checked.add(office)`: what the agent remembers |
| P01 | generator | `yield office`: values one at a time, usable once |
| P02a | match a variable | `bindings[p] = tokens[i:end]` |
| P02a | rewrite until stable | `if new == e: return new` |
| P03a | forward chaining | `if head not in known and all(b in known for b in body)` |
| P03a | backward chaining | `all(backward_chain(rules, facts, b, ...) for b in body)` |
| P03a | loop guard | `if goal in seen: return False` |
| P03a | diagnosis | `all(set(group) & c for c in conflicts)`: hit every conflict |
| P04a | BFS vs DFS | `frontier.pop(0)` vs `frontier.pop()` |
| P04a | repeated states | `if state in visited: continue` |
| P05a | UCS / greedy / A-star | `return g` / `return h` / `return g + weight * h` |
| P05a | branch and bound | `if ng + heuristic(nxt, goal) >= best['cost']: prune` |
| P07 | minimax | `max(...)` for X, `min(...)` for O |
| P07 | alpha-beta | `if beta <= alpha: break` |
| P07 | depth limit | `if depth == 0: return evaluate(board)` |

## Observations

Fill these in from **your own** output. They are the numbers an examiner is most likely to
ask about.

| Part | Quantity | Your value |
|---|---|---|
| A | actions taken with the belief state, starting at office 2 | 11 |
| A | parcels left behind by the agent with no memory | 1 (the parcel in office 1) |
| C | rules examined: forward vs backward, for `upstairs_delivery` | forward 16, backward 30 |
| C | root cause of `upstairs_delivery` failing | `lift_card` is not a fact and no rule concludes it |
| C | conflicts / diagnoses for the two fault reports | 2 conflicts, 2 diagnoses (each of size 2) |
| D | BFS steps and cost to the admin block | 20 steps, cost 30 |
| D | DFS steps and cost to the admin block | 32 steps, cost 42 |
| E | UCS / greedy / A-star cost and nodes expanded | UCS 28 and 63 nodes; greedy 30 and 23 nodes; A-star 28 and 50 nodes |
| E | smallest weight that lost optimality | 2.0 (cost became 30 instead of 28) |
| E | branch and bound: bounds, nodes expanded | bounds 44, 36, 28; 108 nodes expanded (8 branches pruned) |
| F | nodes: plain minimax / alpha-beta normal / alpha-beta reversed | 150 / 83 / 37 |
| F | move chosen at depth 1 and at depth 2 | depth 1 chose cell 4; depth 2 chose cell 6 |

## Viva questions for the practical exam

These mix all six practicals, as the examiner will. Answer each in one or two sentences.

1. What is in Raahi's belief state in Part A, and what goes wrong without it?
2. What does a pattern matcher return when it fails, and what does it return for a match with no variables?
3. Point to the line where your inference engine mentions a specific rule.
4. Is "cannot be proved" the same as "false"? Give an example from Part C.
5. What is the difference between a conflict and a diagnosis?
6. Show me the one line that turns breadth-first search into depth-first search.
7. Why did BFS not find the cheapest route to the admin block?
8. Give the one-sentence argument that the grid distance never overestimates on this campus.
9. If h is zero everywhere, which algorithm have you built?
10. Your pruned game search returned a different move from plain minimax. What does that mean?
11. Why does move ordering change the work of alpha-beta but never its answer?
12. Why did the depth-1 player lose a game that the depth-2 player won?

**My answers — Viva questions**

1. It holds the `checked` set (offices seen empty) and the `direction`. Without it Raahi only reacts to the current percept and never stops.
2. It returns `None` when it fails; for a match with no variables it returns an empty dictionary `{}`.
3. There is none. The engine loops over the rules as data and never writes a rule name.
4. No. `upstairs_delivery` could not be proved because `lift_card` is not a fact, but that does not make it false.
5. A conflict is a set of assumptions that cannot all hold; a diagnosis is a smallest set to give up that breaks every conflict.
6. Change `frontier.pop(0)` to `frontier.pop()`.
7. BFS counts steps, not cost. It crossed the lawn (cost 30), while the route round the lawn costs 28.
8. Each step moves one cell, so the real route needs at least as many steps as rows plus columns to go, and each step costs at least 1.
9. Uniform-cost search (lowest-cost-first).
10. That means there is a bug. Alpha-beta must return the same value and move as minimax.
11. Alpha-beta cuts more when the best moves are looked at first, but it only skips branches that cannot change the decision.
12. At depth 1 it could not see O's reply, so it trusted a weak evaluation; depth 2 saw the threat.

## Before the exam: checklist

- [ ] The notebook runs top to bottom from a clean kernel (Kernel → Restart & Run All)
- [ ] Every `___` blank is filled, and your output matches every **✅ You should see** box
- [ ] I can answer every **Check yourself** question without looking at the code
- [ ] I can point to each line in the one-page summary and say what it does
- [ ] The observations table is filled in from my own output
- [ ] I have tried the viva questions out loud with a classmate

In [68]:
NAME = "Husnain"
ROLL_NUMBER = "2025_cse_064"
print(NAME, ROLL_NUMBER)

Husnain 2025_cse_064
